# Introduction
This notebook is used to fetch, plot, and analyze experiment results.

## 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 39.3936


In [2]:
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability
Initializing src package
Initializing src package


## 2. Fetch Results

In [3]:
import seml
import pandas as pd

# db_collection = 'llama-typo-eval'
db_collection = 'llama-quant-eval'
#states=["FAILED"]
states = ["COMPLETED"]

all_results = seml.evaluation.get_results(db_collection, to_data_frame=True, states=states)
print(f"Lenght of all_results: {len(all_results)}")
print(all_results.columns)

all_results.head()

Output()

Output()

Lenght of all_results: 60
Index(['_id', 'config.overwrite', 'config.db_collection',
       'config.dataset_name', 'config.device', 'config.exp_id',
       'config.max_entries', 'config.max_new_tokens', 'config.model_name',
       'config.n_beams', 'config.n_repeats', 'config.num_excel_rows',
       'config.save_excel', 'config.seed', 'config.strategy',
       'config.temperature', 'config.typo_intensity', 'config.typo_type',
       'config.use_beam_search', 'result.AUCROC_sample', 'result.AUCPR_sample',
       'result.Brier_sample', 'result.LogLoss_sample', 'result.Entropy_sample',
       'result.AUCROC_adj', 'result.AUCPR_adj', 'result.Brier_adj',
       'result.LogLoss_adj', 'result.Entropy_adj', 'result.AUCROC_sem',
       'result.AUCPR_sem', 'result.Brier_sem', 'result.LogLoss_sem',
       'result.Entropy_sem', 'result.Accuracy', 'result.fail_trace'],
      dtype='object')


,_id,config.overwrite,config.db_collection,config.dataset_name,config.device,config.exp_id,config.max_entries,config.max_new_tokens,config.model_name,config.n_beams,...,result.Brier_adj,result.LogLoss_adj,result.Entropy_adj,result.AUCROC_sem,result.AUCPR_sem,result.Brier_sem,result.LogLoss_sem,result.Entropy_sem,result.Accuracy,result.fail_trace
0,1,1,llama-quant-eval,P17,cuda,typo-quant-test-10-03,None,25,Llama-3-8B-AWQ-4bit-local,5,...,0.267428,2.990566,34.977785,1.0,1.0,0.0,2.220446e-16,-9.579496e-08,0.713978,<function get_results at 0x7efef2cb7f40>
1,2,2,llama-quant-eval,P17,cuda,typo-quant-test-10-03,None,25,Llama-3-8B-AWQ-4bit-local,5,...,0.267428,2.990566,34.977785,1.0,1.0,0.0,2.220446e-16,-9.579496e-08,0.713978,<function get_results at 0x7efef2cb7f40>
2,3,3,llama-quant-eval,P17,cuda,typo-quant-test-10-03,None,25,Llama-3-8B-AWQ-4bit-local,5,...,0.267428,2.990566,34.977785,1.0,1.0,0.0,2.220446e-16,-9.579496e-08,0.713978,<function get_results at 0x7efef2cb7f40>
3,4,4,llama-quant-eval,P17,cuda,typo-quant-test-10-03,None,25,Llama-3-8B-AWQ-4bit-local,5,...,0.335173,3.400959,39.891361,1.0,1.0,0.0,2.220446e-16,-8.555182e-08,0.637634,<function get_results at 0x7efef2cb7f40>
4,5,5,llama-quant-eval,P17,cuda,typo-quant-test-10-03,None,25,Llama-3-8B-AWQ-4bit-local,5,...,0.398785,4.715126,41.803758,1.0,1.0,0.0,2.220446e-16,-7.660711e-08,0.570968,<function get_results at 0x7efef2cb7f40>


In [4]:
nan_perplexity_df = all_results[all_results['result.perplexity'].isna() & all_results['config.overwrite'].notna()]
print(nan_perplexity_df['config.quantize_method'])

Series([], Name: config.quantize_method, dtype: object)


In [5]:
columns_to_remove = [
    'config.overwrite',
    'config.db_collection',
    'config.clean_cache',
    'config.device',
    'config.quantized_model_save_path',
    'config.save_quantized_model',
    'config.seed',
    'result',
    'result.current_gpu_type',
    'result.current_gpu_total_memory',
    'result.fail_trace'
]

all_results = all_results.drop(columns=columns_to_remove, errors='ignore')
all_results.columns

Index(['_id', 'config.batch_size', 'config.calib_dataset_name',
       'config.calib_dataset_split', 'config.calib_seq_length',
       'config.eval_dataset_name', 'config.eval_dataset_split',
       'config.eval_metrics', 'config.eval_n_samples',
       'config.eval_seq_length', 'config.model_name', 'config.quantize_method',
       'result.current_gpu_free_memory', 'result.perplexity',
       'result.brier_score', 'result.disk_space_usage',
       'result.quantize_runtime'],
      dtype='object')

In [6]:
all_results.head()

,_id,config.batch_size,config.calib_dataset_name,config.calib_dataset_split,config.calib_seq_length,config.eval_dataset_name,config.eval_dataset_split,config.eval_metrics,config.eval_n_samples,config.eval_seq_length,config.model_name,config.quantize_method,result.current_gpu_free_memory,result.perplexity,result.brier_score,result.disk_space_usage,result.quantize_runtime
0,1,1,WikiText,validation,2048,WikiText,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,Llama-3-8B,NONE,[23.896484375],5.616476,0.000004,nan,nan
1,2,1,WikiText,validation,2048,WikiText,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,Llama-3-8B,BNB-4,[31.123046875],6.307941,0.000004,15.14 GB,15.353248
2,3,1,WikiText,validation,2048,WikiText,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,Llama-3-8B,BNB-8,[28.4453125],5.697184,0.000004,10.42 GB,26.787766
3,4,1,WikiText,validation,2048,WikiText,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,Llama-3-8B,AWQ-4,[35.19140625],8.559265,0.000005,5.33 GB,994.555119
4,5,1,WikiText,validation,2048,WikiText,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,Llama-3-8B,HQQ-8-uniform,[29.294921875],5.615966,0.000004,nan,11.029122


In [7]:
from src.algorithms.quantization import QUANT_CONFIGS

# Define a function to label datasets explicitly
def label_dataset(row):
    return f"Calib: {row['config.calib_dataset_name']} ({row['config.calib_dataset_split']}) | Eval: {row['config.eval_dataset_name']} ({row['config.eval_dataset_split']})"

# Adding new columns to check if splits and datasets are the same
all_results['split_equals'] = all_results['config.calib_dataset_split'] == all_results['config.eval_dataset_split']
all_results['dataset_equals'] = all_results['config.calib_dataset_name'] == all_results['config.eval_dataset_name']
all_results['dataset_label'] = all_results.apply(label_dataset, axis=1)

# Function to extract quantize_method and n_bits
def extract_quantize_method_and_bits(quantize_method):
    config = QUANT_CONFIGS[quantize_method]
    quantize_method = config['quantize_method']
    n_bits = config.get('num_bits', None)
    return quantize_method, n_bits

# Apply the function to create new columns
all_results[['config.quantize_method_type', 'config.n_bits']] = all_results['config.quantize_method'].apply(
    lambda x: pd.Series(extract_quantize_method_and_bits(x))
)

# Convert disk space usage from string (e.g., '15.14 GB') to float (e.g., 15.14)
all_results['result.disk_space_usage'] = all_results['result.disk_space_usage'].str.replace(' GB', '').astype(float)

# Rename the column
all_results.head()

,_id,config.batch_size,config.calib_dataset_name,config.calib_dataset_split,config.calib_seq_length,config.eval_dataset_name,config.eval_dataset_split,config.eval_metrics,config.eval_n_samples,config.eval_seq_length,...,result.current_gpu_free_memory,result.perplexity,result.brier_score,result.disk_space_usage,result.quantize_runtime,split_equals,dataset_equals,dataset_label,config.quantize_method_type,config.n_bits
0,1,1,WikiText,validation,2048,WikiText,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,[23.896484375],5.616476,0.000004,NaN,nan,False,True,Calib: WikiText (validation) | Eval: WikiText ...,NONE,16
1,2,1,WikiText,validation,2048,WikiText,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,[31.123046875],6.307941,0.000004,15.14,15.353248,False,True,Calib: WikiText (validation) | Eval: WikiText ...,BNB,4
2,3,1,WikiText,validation,2048,WikiText,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,[28.4453125],5.697184,0.000004,10.42,26.787766,False,True,Calib: WikiText (validation) | Eval: WikiText ...,BNB,8
3,4,1,WikiText,validation,2048,WikiText,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,[35.19140625],8.559265,0.000005,5.33,994.555119,False,True,Calib: WikiText (validation) | Eval: WikiText ...,AWQ,4
4,5,1,WikiText,validation,2048,WikiText,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,[29.294921875],5.615966,0.000004,NaN,11.029122,False,True,Calib: WikiText (validation) | Eval: WikiText ...,HQQ,8


In [13]:
all_results.loc[all_results['config.quantize_method_type'] == 'QUANTO']

,_id,config.batch_size,config.calib_dataset_name,config.calib_dataset_split,config.calib_seq_length,config.eval_dataset_name,config.eval_dataset_split,config.eval_metrics,config.eval_n_samples,config.eval_seq_length,...,result.perplexity,result.brier_score,result.disk_space_usage,result.quantize_runtime,split_equals,dataset_equals,dataset_label,config.quantize_method_type,config.n_bits,result.disk_space_usage_float
7,8,1,WikiText,validation,2048,WikiText,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,inf,0.000008,nan,5.387069,False,True,Calib: WikiText (validation) | Eval: WikiText ...,QUANTO,8,NaN
16,20,1,WikiText,validation,2048,OpenAssistant,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,inf,0.000008,nan,4.634208,False,False,Calib: WikiText (validation) | Eval: OpenAssis...,QUANTO,8,NaN
25,32,1,WikiText,validation,2048,C4,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,inf,0.000008,nan,4.544218,False,False,Calib: WikiText (validation) | Eval: C4 (test),QUANTO,8,NaN
34,44,1,WikiText,validation,2048,PTB,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,inf,0.000008,nan,5.346802,False,False,Calib: WikiText (validation) | Eval: PTB (test),QUANTO,8,NaN
43,56,1,OpenAssistant,validation,2048,WikiText,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,inf,0.000008,nan,5.698889,False,False,Calib: OpenAssistant (validation) | Eval: Wiki...,QUANTO,8,NaN
52,68,1,OpenAssistant,validation,2048,OpenAssistant,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,inf,0.000008,nan,5.750916,False,True,Calib: OpenAssistant (validation) | Eval: Open...,QUANTO,8,NaN
61,80,1,OpenAssistant,validation,2048,C4,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,inf,0.000008,nan,4.839127,False,False,Calib: OpenAssistant (validation) | Eval: C4 (...,QUANTO,8,NaN
70,92,1,OpenAssistant,validation,2048,PTB,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,inf,0.000008,nan,6.161056,False,False,Calib: OpenAssistant (validation) | Eval: PTB ...,QUANTO,8,NaN
79,104,1,C4,validation,2048,WikiText,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,inf,0.000008,nan,7.37031,False,False,Calib: C4 (validation) | Eval: WikiText (test),QUANTO,8,NaN
88,116,1,C4,validation,2048,OpenAssistant,test,"[perplexity, brier_score, disk_space_usage, qu...",None,2048,...,inf,0.000008,nan,4.780027,False,False,Calib: C4 (validation) | Eval: OpenAssistant (...,QUANTO,8,NaN


In [14]:
all_results['config.n_bits']

0      16
1       4
2       8
3       4
4       8
       ..
139     8
140     4
141     4
142     8
143     2
Name: config.n_bits, Length: 144, dtype: int64

## 3. Plot results

In [4]:
all_results["config.model_name"]

0     Llama-3-8B
1     Llama-3-8B
2     Llama-3-8B
3     Llama-3-8B
4     Llama-3-8B
         ...    
57    Llama-3-8B
58    Llama-3-8B
59    Llama-3-8B
60    Llama-3-8B
61    Llama-3-8B
Name: config.model_name, Length: 62, dtype: object

## 3.1 Experimentation

In [17]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import os
from fpdf import FPDF

def create_line_plot(df, metric):
    fig = go.Figure()
    for typo_type in df['config.typo_type'].unique():
        df_type = df[df['config.typo_type'] == typo_type]
        fig.add_trace(go.Scatter(
            x=df_type['config.typo_intensity'],
            y=df_type[f'result.{metric}'],
            mode='lines+markers',
            name=typo_type
        ))
    
    fig.update_layout(
        title=f'{metric} vs Typo Intensity for Different Typo Types',
        xaxis_title='Typo Intensity',
        yaxis_title=metric,
        legend_title='Typo Type',
        height=800,
        width=1200
    )
    return fig

def create_radar_plot(df, metric):
    fig = go.Figure()
    
    for intensity in df['config.typo_intensity'].unique():
        df_intensity = df[df['config.typo_intensity'] == intensity]
        fig.add_trace(go.Scatterpolar(
            r=df_intensity[f'result.{metric}'],
            theta=df_intensity['config.typo_type'],
            fill='toself',
            name=f'Intensity {intensity}'
        ))
    
    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[df[f'result.{metric}'].min(), df[f'result.{metric}'].max()]
            )),
        showlegend=True,
        title=f'Radar Plot of {metric} for Different Typo Types and Intensities',
        height=800,
        width=1200
    )
    return fig

def create_box_plot(df, metric):
    fig = go.Figure()
    
    for typo_type in df['config.typo_type'].unique():
        df_type = df[df['config.typo_type'] == typo_type]
        fig.add_trace(go.Box(
            y=df_type[f'result.{metric}'],
            x=df_type['config.typo_intensity'],
            name=typo_type,
            boxpoints='all'
        ))
    
    fig.update_layout(
        title=f'Box Plot of {metric} for Different Typo Types and Intensities',
        xaxis_title='Typo Intensity',
        yaxis_title=metric,
        boxmode='group',
        height=800,
        width=1200
    )
    return fig

def create_bubble_plot(df, metric):
    fig = px.scatter(df, x='config.typo_type', y=f'result.{metric}',
                     size='config.typo_intensity', color='config.typo_type',
                     hover_name='config.typo_type', size_max=60)
    
    fig.update_layout(
        title=f'Bubble Plot of {metric} for Different Typo Types and Intensities',
        xaxis_title='Typo Type',
        yaxis_title=metric,
        height=800,
        width=1200
    )
    return fig

def create_parallel_coordinates_plot(df):
    # Create a color map for typo types
    typo_types = df['config.typo_type'].unique()
    color_map = px.colors.qualitative.Plotly[:len(typo_types)]
    color_dict = dict(zip(typo_types, color_map))
    
    # Create a new column with color values
    df['color'] = df['config.typo_type'].map(color_dict)
    
    fig = go.Figure(data=
        go.Parcoords(
            line = dict(color = df['color'],
                        colorscale = color_map,
                        showscale = True),
            dimensions = list([
                dict(range = [df['config.typo_intensity'].min(), df['config.typo_intensity'].max()],
                     label = 'Typo Intensity', values = df['config.typo_intensity']),
                dict(range = [df['result.Accuracy'].min(), df['result.Accuracy'].max()],
                     label = 'Accuracy', values = df['result.Accuracy']),
                dict(range = [df['result.AUCPR_sem'].min(), df['result.AUCPR_sem'].max()],
                     label = 'AUCPR_sem', values = df['result.AUCPR_sem'])
            ])
        )
    )
    
    fig.update_layout(
        title='Parallel Coordinates Plot of Metrics for Different Typo Types and Intensities',
        height=800,
        width=1200
    )
    
    # Add a legend
    for typo_type, color in color_dict.items():
        fig.add_trace(go.Scatter(
            x=[None], y=[None], 
            mode='markers',
            marker=dict(size=10, color=color),
            legendgroup=typo_type,
            showlegend=True,
            name=typo_type
        ))
    
    return fig

def generate_pdf_report(plots, pdf_path, model_name, strategy, max_new_tokens, temperature):
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    
    # Add title page
    pdf.add_page()
    pdf.set_font("Arial", 'B', size=16)
    pdf.cell(0, 10, "Typo Effect Analysis", ln=True, align='C')
    pdf.set_font("Arial", size=12)
    pdf.cell(0, 10, f"Model: {model_name}", ln=True, align='C')
    pdf.cell(0, 10, f"Strategy: {strategy}", ln=True, align='C')
    pdf.cell(0, 10, f"Max New Tokens: {max_new_tokens}", ln=True, align='C')
    pdf.cell(0, 10, f"Temperature: {temperature}", ln=True, align='C')

    # Add plots to the PDF
    for plot_file, desc in plots:
        pdf.add_page()
        pdf.set_font("Arial", 'B', size=14)
        pdf.multi_cell(0, 10, desc)
        pdf.ln(5)
        pdf.image(plot_file, w=pdf.w - 20)
        
        pdf.ln(10)
        pdf.set_font("Arial", size=10)
        pdf.multi_cell(0, 5, "This plot illustrates the impact of different typo types and intensities on " 
                             "the model's performance. Variations across typo types and intensities " 
                             "indicate areas where the model's reliability may be affected.")

    pdf.output(pdf_path, "F")

def main(all_results):
    # Create a directory for saving plots
    plots_dir = "plots/typo_effect_analysis"
    os.makedirs(plots_dir, exist_ok=True)

    plots = []
    metrics = ['Accuracy', 'AUCPR_sem']
    plot_types = ['Line Plot', 'Radar Plot', 'Box Plot', 'Bubble Plot']
    plot_functions = [create_line_plot, create_radar_plot, create_box_plot, create_bubble_plot]

    for metric in metrics:
        for plot_type, plot_function in zip(plot_types, plot_functions):
            fig = plot_function(all_results, metric)
            plot_file = os.path.join(plots_dir, f"{metric}_{plot_type.replace(' ', '_')}.png")
            fig.write_image(plot_file)
            plots.append((plot_file, f"{plot_type} of {metric}"))
    
    # Add parallel coordinates plot
    fig = create_parallel_coordinates_plot(all_results)
    plot_file = os.path.join(plots_dir, "Parallel_Coordinates_Plot.png")
    fig.write_image(plot_file)
    plots.append((plot_file, "Parallel Coordinates Plot of Metrics"))

    # Generate the PDF report
    model_name = all_results['config.model_name'].iloc[0]
    strategy = all_results['config.strategy'].iloc[0]
    max_new_tokens = all_results['config.max_new_tokens'].iloc[0]
    temperature = all_results['config.temperature'].iloc[0]

    pdf_path = os.path.join(plots_dir, "typo_effect_analysis_report.pdf")
    generate_pdf_report(plots, pdf_path, model_name, strategy, max_new_tokens, temperature)

    print(f"PDF report generated and saved as '{pdf_path}'")

if __name__ == "__main__":
    # Load your results dataframe here
    # all_results = pd.read_csv('your_results.csv')  # or however you load your results
    
    # Call the main function
    main(all_results_cleaned)

/tmp/ipykernel_3520843/108234446.py:97: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



PDF report generated and saved as 'plots/typo_effect_analysis/typo_effect_analysis_report.pdf'


In [27]:
import pandas as pd
import plotly.graph_objects as go
import os
from fpdf import FPDF
import plotly.express as px
import numpy as np

def create_radar_plot_1(df, metric):
    fig = go.Figure()
    
    for intensity in df['config.typo_intensity'].unique():
        df_intensity = df[df['config.typo_intensity'] == intensity]
        values = df_intensity[f'result.{metric}'].tolist()
        values.append(values[0])  # Connect last point to first
        
        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=df_intensity['config.typo_type'].tolist() + [df_intensity['config.typo_type'].iloc[0]],
            fill='toself',
            name=f'Intensity {intensity}',
            line=dict(color=px.colors.sequential.Viridis[intensity])
        ))
    
    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[df[f'result.{metric}'].min(), df[f'result.{metric}'].max()]
            )),
        showlegend=True,
        title=f'Radar Plot of {metric} (Viridis Color Scheme)',
        height=800,
        width=1200
    )
    return fig

def create_radar_plot_2(df, metric):
    fig = go.Figure()
    
    color_scale = px.colors.sequential.Reds
    for i, intensity in enumerate(df['config.typo_intensity'].unique()):
        df_intensity = df[df['config.typo_intensity'] == intensity]
        values = df_intensity[f'result.{metric}'].tolist()
        values.append(values[0])  # Connect last point to first
        
        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=df_intensity['config.typo_type'].tolist() + [df_intensity['config.typo_type'].iloc[0]],
            fill='toself',
            name=f'Intensity {intensity}',
            line=dict(color=color_scale[i*3])  # Use every third color for more contrast
        ))
    
    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[df[f'result.{metric}'].min(), df[f'result.{metric}'].max()]
            )),
        showlegend=True,
        title=f'Radar Plot of {metric} (Red Tones)',
        height=800,
        width=1200
    )
    return fig

def create_radar_plot_3(df, metric):
    fig = go.Figure()
    
    for intensity in df['config.typo_intensity'].unique():
        df_intensity = df[df['config.typo_intensity'] == intensity]
        values = df_intensity[f'result.{metric}'].tolist()
        values.append(values[0])  # Connect last point to first
        
        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=df_intensity['config.typo_type'].tolist() + [df_intensity['config.typo_type'].iloc[0]],
            fill='toself',
            name=f'Intensity {intensity}',
            line=dict(color=px.colors.diverging.RdBu[intensity * 2])
        ))
    
    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[df[f'result.{metric}'].min(), df[f'result.{metric}'].max()]
            )),
        showlegend=True,
        title=f'Radar Plot of {metric} (Diverging Color Scheme)',
        height=800,
        width=1200
    )
    return fig

def create_radar_plot_4(df, metric):
    fig = go.Figure()
    
    for intensity in df['config.typo_intensity'].unique():
        df_intensity = df[df['config.typo_intensity'] == intensity]
        values = df_intensity[f'result.{metric}'].tolist()
        values.append(values[0])  # Connect last point to first
        
        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=df_intensity['config.typo_type'].tolist() + [df_intensity['config.typo_type'].iloc[0]],
            fill='none',
            name=f'Intensity {intensity}',
            line=dict(color=px.colors.qualitative.Pastel[intensity], width=2)
        ))
    
    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[df[f'result.{metric}'].min(), df[f'result.{metric}'].max()]
            )),
        showlegend=True,
        title=f'Radar Plot of {metric} (Pastel Colors, No Fill)',
        height=800,
        width=1200
    )
    return fig

def create_radar_plot_5(df, metric):
    fig = go.Figure()
    
    for intensity in df['config.typo_intensity'].unique():
        df_intensity = df[df['config.typo_intensity'] == intensity]
        values = df_intensity[f'result.{metric}'].tolist()
        values.append(values[0])  # Connect last point to first
        
        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=df_intensity['config.typo_type'].tolist() + [df_intensity['config.typo_type'].iloc[0]],
            fill='toself',
            name=f'Intensity {intensity}',
            line=dict(color='rgba(0,0,0,0)'),
            fillcolor=px.colors.sequential.Plasma[intensity]
        ))
    
    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[df[f'result.{metric}'].min(), df[f'result.{metric}'].max()]
            )),
        showlegend=True,
        title=f'Radar Plot of {metric} (Plasma Color Scheme, Only Fill)',
        height=800,
        width=1200
    )
    return fig

def create_radar_plot_6(df, metric):
    fig = go.Figure()
    
    for intensity in df['config.typo_intensity'].unique():
        df_intensity = df[df['config.typo_intensity'] == intensity]
        values = df_intensity[f'result.{metric}'].tolist()
        values.append(values[0])  # Connect last point to first
        
        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=df_intensity['config.typo_type'].tolist() + [df_intensity['config.typo_type'].iloc[0]],
            fill='toself',
            name=f'Intensity {intensity}',
            line=dict(color=px.colors.cyclical.Edge[intensity]),
            opacity=0.7
        ))
    
    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[df[f'result.{metric}'].min(), df[f'result.{metric}'].max()]
            )),
        showlegend=True,
        title=f'Radar Plot of {metric} (Cyclical Color Scheme with Transparency)',
        height=800,
        width=1200
    )
    return fig

def create_radar_plot_7(df, metric):
    fig = go.Figure()
    
    for intensity in df['config.typo_intensity'].unique():
        df_intensity = df[df['config.typo_intensity'] == intensity]
        values = df_intensity[f'result.{metric}'].tolist()
        values.append(values[0])  # Connect last point to first
        
        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=df_intensity['config.typo_type'].tolist() + [df_intensity['config.typo_type'].iloc[0]],
            fill='toself',
            name=f'Intensity {intensity}',
            line=dict(color='rgb(0,0,0)', width=1),
            fillcolor=f'rgba(0, 0, 0, {(intensity + 1) / 10})'  # Increasing opacity for higher intensities
        ))
    
    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[df[f'result.{metric}'].min(), df[f'result.{metric}'].max()]
            )),
        showlegend=True,
        title=f'Radar Plot of {metric} (Grayscale with Increasing Opacity)',
        height=800,
        width=1200
    )
    return fig

def create_radar_plot_8(df, metric):
    fig = go.Figure()
    
    intensities = df['config.typo_intensity'].unique()
    max_intensity = intensities.max()
    
    for intensity in intensities:
        df_intensity = df[df['config.typo_intensity'] == intensity]
        values = df_intensity[f'result.{metric}'].tolist()
        values.append(values[0])  # Connect last point to first
        
        # Calculate marker size based on intensity
        marker_size = 5 + (intensity / max_intensity) * 15
        
        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=df_intensity['config.typo_type'].tolist() + [df_intensity['config.typo_type'].iloc[0]],
            fill='none',
            name=f'Intensity {intensity}',
            line=dict(color=px.colors.sequential.Burgyl[intensity], width=2),
            marker=dict(size=marker_size, symbol='circle')
        ))
    
    fig.update_layout(
        polar=dict(
            radialaxis=dict(
                visible=True,
                range=[df[f'result.{metric}'].min(), df[f'result.{metric}'].max()]
            )),
        showlegend=True,
        title=f'Radar Plot of {metric} (Burgyl Color Scheme with Varying Marker Sizes)',
        height=800,
        width=1200
    )
    return fig

def generate_pdf_report(plots, pdf_path, model_name, strategy, max_new_tokens, temperature):
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    
    # Add title page
    pdf.add_page()
    pdf.set_font("Arial", 'B', size=16)
    pdf.cell(0, 10, "Typo Effect Analysis", ln=True, align='C')
    pdf.set_font("Arial", size=12)
    pdf.cell(0, 10, f"Model: {model_name}", ln=True, align='C')
    pdf.cell(0, 10, f"Strategy: {strategy}", ln=True, align='C')
    pdf.cell(0, 10, f"Max New Tokens: {max_new_tokens}", ln=True, align='C')
    pdf.cell(0, 10, f"Temperature: {temperature}", ln=True, align='C')

    # Add plots to the PDF
    for plot_file, desc in plots:
        pdf.add_page()
        pdf.set_font("Arial", 'B', size=14)
        pdf.multi_cell(0, 10, desc)
        pdf.ln(5)
        pdf.image(plot_file, w=pdf.w - 20)
        
        pdf.ln(10)
        pdf.set_font("Arial", size=10)
        pdf.multi_cell(0, 5, "This radar plot illustrates the impact of different typo types and intensities on " 
                             "the model's performance. Each axis represents a typo type, while different intensities "
                             "are shown as separate traces. The plot allows for easy comparison of performance "
                             "across different typo types and intensities.")

    pdf.output(pdf_path, "F")

def main(all_results):
    # Create a directory for saving plots
    plots_dir = "plots/typo_effect_analysis"
    os.makedirs(plots_dir, exist_ok=True)

    plots = []
    metrics = ['Accuracy', 'AUCPR_sem']
    plot_functions = [create_radar_plot_1, create_radar_plot_2, create_radar_plot_3, create_radar_plot_4,
                      create_radar_plot_5, create_radar_plot_6, create_radar_plot_7, create_radar_plot_8]

    for metric in metrics:
        for i, plot_function in enumerate(plot_functions, 1):
            fig = plot_function(all_results, metric)
            plot_file = os.path.join(plots_dir, f"{metric}_Radar_Plot_{i}.png")
            fig.write_image(plot_file)
            plots.append((plot_file, f"Radar Plot {i} of {metric}"))

    # Generate the PDF report
    model_name = all_results['config.model_name'].iloc[0]
    strategy = all_results['config.strategy'].iloc[0]
    max_new_tokens = all_results['config.max_new_tokens'].iloc[0]
    temperature = all_results['config.temperature'].iloc[0]

    pdf_path = os.path.join(plots_dir, "typo_effect_analysis_report.pdf")
    generate_pdf_report(plots, pdf_path, model_name, strategy, max_new_tokens, temperature)

    print(f"PDF report generated and saved as '{pdf_path}'")

if __name__ == "__main__":
    # Load your results dataframe here
    # all_results = pd.read_csv('your_results.csv')  # or however you load your results
    
    # Call the main function
    main(all_results_cleaned)

PDF report generated and saved as 'plots/typo_effect_analysis/typo_effect_analysis_report.pdf'


## 3.2 Final plots

In [8]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from fpdf import FPDF
import numpy as np

# Define custom colors
color_A = '#7FFFD4'  # Aquamarine
color_B = '#FF69B4'  # Hot Pink
color_C = '#FFD700'  # Gold
color_D = '#1E90FF'  # Dodger Blue

# Define color schemes
color_schemes = [
    [color_B, color_D, color_C],
]

bar_colors = ['#1f77b4', '#2ca02c', '#d62728', '#ff7f0e']  # Blue, Green, Red, Orange

def create_radar_plot(df, metric):
    fig = go.Figure()
    
    intensities = df['config.typo_intensity'].unique()
    for i, intensity in enumerate(intensities):
        df_intensity = df[df['config.typo_intensity'] == intensity]
        values = df_intensity[f'result.{metric}'].tolist()
        values.append(values[0])  # Connect last point to first
        
        color = color_schemes[0][i]  # Use the first (and only) color scheme
        rgb = tuple(int(color.lstrip('#')[j:j+2], 16) for j in (0, 2, 4))
        rgba_fill = f'rgba({rgb[0]}, {rgb[1]}, {rgb[2]}, 0.5)'  # 0.5 for 50% transparency
        
        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=df_intensity['config.typo_type'].tolist() + [df_intensity['config.typo_type'].iloc[0]],
            fill='toself',
            name=f'Intensity {intensity}',
            line=dict(color=color, width=2),
            fillcolor=rgba_fill
        ))
    
    # Add baseline circle
    baseline_value = df[df['config.typo_type'] == 'none'][f'result.{metric}'].iloc[0]
    fig.add_trace(go.Scatterpolar(
        r=[baseline_value] * (len(df['config.typo_type'].unique()) + 1),
        theta=df['config.typo_type'].unique().tolist() + [df['config.typo_type'].unique()[0]],
        name='Baseline',
        line=dict(color='black', width=2, dash='dash'),
    ))
    
    # Adjust range for entropy metrics
    if 'Entropy' in metric:
        max_value = df[f'result.{metric}'].max()
        range_value = [0, max_value * 1.1]  # Add 10% padding
    else:
        range_value = [0, 1]
    
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=range_value)),
        showlegend=True,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        title=dict(text=f'Radar Plot of {metric}', font=dict(size=18)),
        height=800, width=1000
    )
    return fig

def create_grouped_bar_plots(df, metric):
    groups = [
        ['char_insertion', 'char_deletion', 'char_replacement', 'char_repetition'],
        ['char_swapping', 'char_LCC', 'char_insert_noise', 'char_substitution'],
        ['word_CMW', 'word_remove_punctuation', 'word_internet_slang', 'word_emoji'],
        ['word_synonym', 'word_phrase_translation', 'word_context_aware_insertion', 'word_keyword_only'],
        ['word_taxonomy_neg', 'word_taxonomy_pos', 'word_repeat']
    ]
    
    fig = make_subplots(rows=3, cols=2, subplot_titles=[f"Group {i+1}" for i in range(5)] + [""],
                        vertical_spacing=0.2, horizontal_spacing=0.1,
                        specs=[[{"secondary_y": True}]*2]*3)
    
    baseline_value = df[df['config.typo_type'] == 'none'][f'result.{metric}'].iloc[0]
    
    for i, group in enumerate(groups):
        row = i // 2 + 1
        col = i % 2 + 1
        
        for j, pert_type in enumerate(group):
            y = []
            for intensity in [1, 2, 3]:
                value = df[(df['config.typo_type'] == pert_type) & (df['config.typo_intensity'] == intensity)][f'result.{metric}'].iloc[0]
                y.append(value)
            
            fig.add_trace(
                go.Bar(
                    x=[1, 2, 3],  # Use actual intensity values
                    y=y,
                    name=pert_type,
                    marker_color=bar_colors[j % len(bar_colors)],
                    showlegend=False
                ),
                row=row, col=col
            )
        
        # Add baseline
        fig.add_shape(
            type="line",
            x0=0.5,
            x1=3.5,
            y0=baseline_value,
            y1=baseline_value,
            line=dict(color="black", width=2, dash="dash"),
            row=row, col=col
        )
        
        # Add legend for this subplot
        for j, pert_type in enumerate(group):
            fig.add_trace(
                go.Scatter(
                    x=[None],
                    y=[None],
                    mode='markers',
                    marker=dict(size=10, color=bar_colors[j % len(bar_colors)]),
                    showlegend=True,
                    name=pert_type,
                    legendgroup=f'group{i}',
                    legendgrouptitle_text=f'Group {i+1}'
                ),
                row=row, col=col, secondary_y=True
            )
    
    fig.update_layout(
        barmode='group',
        height=1800,
        width=1600,
        title=f'Grouped Bar Plots of {metric} by Perturbation Type and Intensity',
        bargap=0.15,
        bargroupgap=0.0
    )
    fig.update_xaxes(title_text="Intensity", tickangle=0, tickmode='array', tickvals=[1, 2, 3], range=[0.5, 3.5])
    fig.update_yaxes(title_text=metric)
    
    return fig

def create_aggregated_bar_plot(df, metric):
    fig = go.Figure()
    
    baseline_value = df[df['config.typo_type'] == 'none'][f'result.{metric}'].iloc[0]
    
    for intensity in [1, 2, 3]:
        avg_value = df[df['config.typo_intensity'] == intensity][f'result.{metric}'].mean()
        fig.add_trace(go.Bar(
            x=[intensity],
            y=[avg_value],
            name=f'Intensity {intensity}',
            marker_color=color_schemes[0][intensity-1]
        ))
    
    fig.add_shape(
        type="line",
        x0=0.5,
        x1=3.5,
        y0=baseline_value,
        y1=baseline_value,
        line=dict(color="black", width=2, dash="dash")
    )
    
    fig.update_layout(
        title=f'Aggregated Bar Plot of {metric} by Intensity',
        xaxis_title="Intensity",
        yaxis_title=metric,
        barmode='group',
        bargap=0.15,
        bargroupgap=0.0,
        height=600,
        width=800
    )
    fig.update_xaxes(tickmode='array', tickvals=[1, 2, 3], range=[0.5, 3.5])
    
    return fig

def create_all_perturbations_bar_plot(df, metric):
    fig = go.Figure()
    
    baseline_value = df[df['config.typo_type'] == 'none'][f'result.{metric}'].iloc[0]
    perturbation_types = df['config.typo_type'].unique()
    perturbation_types = [p for p in perturbation_types if p != 'none']
    
    for intensity in [1, 2, 3]:
        y = []
        for pert_type in perturbation_types:
            value = df[(df['config.typo_type'] == pert_type) & (df['config.typo_intensity'] == intensity)][f'result.{metric}'].iloc[0]
            y.append(value)
        
        fig.add_trace(go.Bar(
            x=perturbation_types,
            y=y,
            name=f'Intensity {intensity}',
            marker_color=color_schemes[0][intensity-1]
        ))
    
    fig.add_shape(
        type="line",
        x0=-0.5,
        x1=len(perturbation_types)-0.5,
        y0=baseline_value,
        y1=baseline_value,
        line=dict(color="black", width=2, dash="dash")
    )
    
    fig.update_layout(
        title=f'All Perturbations Bar Plot of {metric}',
        xaxis_title="Perturbation Type",
        yaxis_title=metric,
        barmode='group',
        height=800,
        width=1600
    )
    fig.update_xaxes(tickangle=45)
    
    return fig

import unidecode

def generate_pdf_report(plots, pdf_path, model_name, strategy, max_new_tokens, temperature):
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    
    # Add title page
    pdf.add_page()
    pdf.set_font("Arial", 'B', size=16)
    pdf.cell(0, 10, "Typo Effect Analysis", ln=True, align='C')
    pdf.set_font("Arial", size=12)
    pdf.cell(0, 10, f"Model: {model_name}", ln=True, align='C')
    pdf.cell(0, 10, f"Strategy: {strategy}", ln=True, align='C')
    pdf.cell(0, 10, f"Max New Tokens: {max_new_tokens}", ln=True, align='C')
    pdf.cell(0, 10, f"Temperature: {temperature}", ln=True, align='C')

    # Add plots and formulas to the PDF
    for plot_file, desc, formula in plots:
        pdf.add_page()
        pdf.set_font("Arial", 'B', size=14)
        pdf.multi_cell(0, 10, unidecode.unidecode(desc))
        pdf.ln(5)
        pdf.image(plot_file, w=pdf.w - 20)
        
        # Add formula explanation
        if formula:
            pdf.ln(10)
            pdf.set_font("Arial", size=12)
            pdf.multi_cell(0, 10, unidecode.unidecode(formula))

    pdf.output(pdf_path, "F")

def main(all_results, exp_id):
    plots_dir = f"plots/typo_effect_analysis_{exp_id}"
    os.makedirs(plots_dir, exist_ok=True)

    plots = []
    metrics = ['Accuracy', 'AUCPR_sample', 'AUCPR_adj', 'Entropy_sample', 'Entropy_adj', 'LogLoss_sample', 'LogLoss_adj', 'Brier_sample', 'Brier_adj']

    formulas = {
        'Accuracy': r"$\text{Accuracy} = \frac{\text{Number of correct predictions}}{\text{Total number of predictions}}$",
        'AUCPR_sample': r"$\text{AUCPR} = \int_0^1 \text{Precision}(\text{Recall}) d\text{Recall}$",
        'AUCPR_adj': r"$\text{AUCPR}_\text{adj} = \int_0^1 \text{Precision}_\text{adj}(\text{Recall}_\text{adj}) d\text{Recall}_\text{adj}$",
        'Entropy_sample': r"$\text{Entropy} = -\sum_{i} p_i \log_2(p_i)$",
        'Entropy_adj': r"$\text{Entropy}_\text{adj} = -\sum_{i} p_{\text{adj},i} \log_2(p_{\text{adj},i})$",
        'LogLoss_sample': r"$\text{LogLoss} = -\frac{1}{N} \sum_{i=1}^N [y_i \log(p_i) + (1 - y_i) \log(1 - p_i)]$",
        'LogLoss_adj': r"$\text{LogLoss}_\text{adj} = -\frac{1}{N} \sum_{i=1}^N [y_i \log(p_{\text{adj},i}) + (1 - y_i) \log(1 - p_{\text{adj},i})]$",
        'Brier_sample': r"$\text{Brier Score} = \frac{1}{N} \sum_{i=1}^N (f_i - o_i)^2$",
        'Brier_adj': r"$\text{Brier Score}_\text{adj} = \frac{1}{N} \sum_{i=1}^N (f_{\text{adj},i} - o_i)^2$"
    }

    for metric in metrics:
        # Radar plot
        fig = create_radar_plot(all_results, metric)
        plot_file = os.path.join(plots_dir, f"{metric}_Radar_Plot_{exp_id}.png")
        fig.write_image(plot_file)
        plots.append((plot_file, f"Radar Plot of {metric}", formulas.get(metric, "")))
        
        # Grouped bar plots
        fig = create_grouped_bar_plots(all_results, metric)
        plot_file = os.path.join(plots_dir, f"{metric}_Grouped_Bar_Plots_{exp_id}.png")
        fig.write_image(plot_file)
        plots.append((plot_file, f"Grouped Bar Plots of {metric}", formulas.get(metric, "")))
        
        # Aggregated bar plot
        fig = create_aggregated_bar_plot(all_results, metric)
        plot_file = os.path.join(plots_dir, f"{metric}_Aggregated_Bar_Plot_{exp_id}.png")
        fig.write_image(plot_file)
        plots.append((plot_file, f"Aggregated Bar Plot of {metric}", formulas.get(metric, "")))
        
        # All perturbations bar plot
        fig = create_all_perturbations_bar_plot(all_results, metric)
        plot_file = os.path.join(plots_dir, f"{metric}_All_Perturbations_Bar_Plot_{exp_id}.png")
        fig.write_image(plot_file)
        plots.append((plot_file, f"All Perturbations Bar Plot of {metric}", formulas.get(metric, "")))

    # Generate the PDF report
    model_name = all_results['config.model_name'].iloc[0]
    strategy = all_results['config.strategy'].iloc[0]
    max_new_tokens = all_results['config.max_new_tokens'].iloc[0]
    temperature = all_results['config.temperature'].iloc[0]

    pdf_path = os.path.join(plots_dir, f"typo_effect_analysis_report_{exp_id}.pdf")
    generate_pdf_report(plots, pdf_path, model_name, strategy, max_new_tokens, temperature)

    print(f"PDF report generated and saved as '{pdf_path}'")

if __name__ == "__main__":
    # Load your results dataframe here
    # Assuming you've already run the seml code to get all_results
    # all_results = ... (your existing code to load the results)
    
    # Specify the experiment ID
    exp_id = "awq-pert-10-03"  # You can change this to any string you want
    
    # Call the main function with the experiment ID
    main(all_results, exp_id)

PDF report generated and saved as 'plots/typo_effect_analysis_awq-pert-10-03/typo_effect_analysis_report_awq-pert-10-03.pdf'


## 4. Remove Duplicates

In [4]:
import pandas as pd

def inspect_and_remove_duplicates(df):
    # Define the columns used for pivoting
    pivot_columns = ['config.typo_type', 'config.typo_intensity']
    
    # Find duplicates in pivot columns
    duplicate_mask = df.duplicated(subset=pivot_columns, keep=False)
    duplicates = df[duplicate_mask]
    
    if duplicates.empty:
        print("No duplicates found in pivot columns.")
        return df
    
    print("Duplicate entries found in pivot columns:")
    print(duplicates[pivot_columns])
    
    print("\nFull rows for duplicate entries:")
    print(duplicates)
    
    # Ask user how to handle duplicates
    print("\nHow would you like to handle these duplicates?")
    print("1: Keep first occurrence")
    print("2: Keep last occurrence")
    print("3: Remove all duplicates")
    print("4: Do nothing (keep all)")
    
    choice = input("Enter your choice (1-4): ")
    
    if choice == '1':
        df_cleaned = df.drop_duplicates(subset=pivot_columns, keep='first')
        print(f"Removed {len(df) - len(df_cleaned)} duplicate rows.")
    elif choice == '2':
        df_cleaned = df.drop_duplicates(subset=pivot_columns, keep='last')
        print(f"Removed {len(df) - len(df_cleaned)} duplicate rows.")
    elif choice == '3':
        df_cleaned = df.drop_duplicates(subset=pivot_columns, keep=False)
        print(f"Removed {len(df) - len(df_cleaned)} duplicate rows.")
    elif choice == '4':
        df_cleaned = df
        print("No rows removed.")
    else:
        print("Invalid choice. No rows removed.")
        df_cleaned = df
    
    return df_cleaned

# Assuming your dataframe is named 'all_results'
all_results_cleaned = inspect_and_remove_duplicates(all_results)

# You can now use all_results_cleaned for further processing

Duplicate entries found in pivot columns:
           config.typo_type  config.typo_intensity
42  word_phrase_translation                      1
43  word_phrase_translation                      2
59  word_phrase_translation                      1
60  word_phrase_translation                      2

Full rows for duplicate entries:
    _id  config.overwrite config.db_collection config.dataset_name  \
42   43                43      llama-typo-eval                 P17   
43   44                44      llama-typo-eval                 P17   
59   61                61      llama-typo-eval                 P17   
60   62                62      llama-typo-eval                 P17   

   config.device    config.exp_id config.max_entries  config.max_new_tokens  \
42          cuda  typo-test-10-01               None                     25   
43          cuda  typo-test-10-01               None                     25   
59          cuda  typo-test-10-01               None                     25   
60